# Modèle « contrôle des camions »

Détecter les camions **surchargés ou hors gabarit** à la sortie de l'usine.

Cinq cellules, à exécuter dans l'ordre. Rien à choisir.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → **T4 GPU***.
Sans GPU, comptez dix fois plus de temps.


## 1 · Installation


In [ ]:
!pip install -q ultralytics roboflow

import torch
from ultralytics import YOLO

print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'AUCUN — activez le T4 dans Exécution > Modifier le type d\'exécution')


## 2 · Télécharger le jeu de données

602 images annotées : 186 camions en infraction, 416 conformes.

`truck_odol` = *Over Dimension Over Load*, le terme réglementaire indonésien
pour un camion hors gabarit ou surchargé. `normal_truck` = camion en règle.

Cette seconde classe est la plus importante : c'est elle qui apprend au modèle
à **se taire** quand tout va bien. C'est exactement ce qui manque au modèle
actuel, qui affirme une infraction sur n'importe quelle image.


In [ ]:
from roboflow import Roboflow

CLE = 'kQle1ihpBmsqoXROy27k'    # votre clé Roboflow

rf = Roboflow(api_key=CLE)
projet = rf.workspace('mochammad-eka').project('overloaded-truck')
jeu = projet.version(max(v.version for v in projet.versions())).download('yolov8')

import yaml
conf = yaml.safe_load(open(f'{jeu.location}/data.yaml'))
print('Dossier  :', jeu.location)
print('Classes  :', conf['names'])


## 3 · Entraîner

30 à 50 minutes sur un T4. Vous pouvez laisser l'onglet en arrière-plan, mais
ne fermez pas le navigateur : Colab arrête la session.

`imgsz=640` n'est pas négociable — c'est la taille attendue par le serveur
SmokeWatch. Toute autre valeur ferait planter la détection une fois déployée.


In [ ]:
modele = YOLO('yolov8n.pt')      # modèle neuf : les anciennes classes n'ont plus cours

modele.train(
    data=f'{jeu.location}/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,          # arrêt si aucun progrès pendant 15 époques
    name='camions',
    # Un camion filmé au portail varie en lumière et en angle, jamais
    # en orientation : on augmente les premiers, pas le dernier.
    hsv_v=0.5,
    degrees=8,
    fliplr=0.5,
    flipud=0.0,
)
print('Terminé.')


## 4 · Lire les résultats

Le chiffre qui compte est le **rappel** sur `truck_odol` : la part des camions
en infraction que le modèle voit réellement.

Un rappel faible signifie que des camions sortent en infraction sans que
personne ne le sache — et ça ne se voit nulle part, puisqu'une absence
d'alerte ressemble à une absence d'infraction.


In [ ]:
meilleur = 'runs/detect/camions/weights/best.pt'
modele = YOLO(meilleur)
m = modele.val(data=f'{jeu.location}/data.yaml', imgsz=640, verbose=False)

print(f"{'classe':<18}{'précision':>11}{'rappel':>9}{'mAP50':>9}")
print('-' * 47)
for i, nom in modele.names.items():
    p, r, ap50, _ = m.box.class_result(i)
    marque = '  ← infraction' if 'odol' in nom or 'overload' in nom else ''
    print(f'{nom:<18}{p:>11.3f}{r:>9.3f}{ap50:>9.3f}{marque}')

print()
print(f'Moyenne mAP50 : {float(m.box.map50):.3f}')
print()
print('Repère : au-dessus de 0,70 de rappel sur la classe infraction, le')
print('modèle est exploitable. En dessous de 0,50, il rate un camion sur deux.')


## 5 · Récupérer le modèle


In [ ]:
from google.colab import files
import shutil

shutil.copy(meilleur, 'smokewatch_load_control_best.pt')
files.download('smokewatch_load_control_best.pt')


---

## Installer le modèle sur le serveur

```powershell
# 1. Garder l'ancien de côté — il faut pouvoir revenir en arrière
move models\smokewatch_load_control_best.pt models\load_control_ancien.pt

# 2. Déposer le nouveau à sa place
copy %USERPROFILE%\Downloads\smokewatch_load_control_best.pt models\

# 3. Supprimer l'ancien export — sinon le pipeline garde l'ancien modèle
rmdir /s /q models\smokewatch_load_control_best_openvino_model

# 4. Reconvertir et redémarrer
.\venv\Scripts\python.exe scripts\export_openvino.py
```

L'étape 3 n'est pas facultative : sans elle, le pipeline continue d'utiliser
l'ancien modèle **sans rien signaler**.

Enfin, dans l'interface → **Paramètres**, réactivez la détection pour
*Contrôle chargement* : elle est désactivée depuis que l'ancien modèle
produisait des fausses alertes.

## Ce que ce modèle saura, et ne saura pas

**Il saura** reconnaître un camion surchargé ou hors gabarit, et se taire
devant un camion conforme.

**Il ne saura pas** dire si un chargement est bâché : aucun jeu public ne
contient de camions *non bâchés* annotés comme tels. Cela viendra avec vos
images de portail — activez la collecte automatique sur la caméra de sortie,
et vous aurez le jeu de données en une semaine sans trier une seule image.
